# 04 - End-to-End RL + MLOps Pipeline

Ten notebook jest ustawiony jako prosty, powtarzalny flow:

1. Inicjacja środowiska
2. Integracja z MLOps (MLflow)
3. RL z logowaniem do MLOps od pierwszego kroku
4. Statystyki i przegląd runow

Uruchamiaj komorki po kolei od góry.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, '../..')

import numpy as np
import pandas as pd
import mlflow
import torch

from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecMonitor, VecFrameStack

from src.environment.pacman_env import PacmanGridEnv
from src.utils.mlflow_logger import MLflowLogger
from src.utils.maskable_env import (
    create_maskable_ppo,
    load_trainable_model,
    wrap_with_action_masker,
    predict_action,
)
from src.utils.obs_sync import include_flags_from_checkpoint

ROOT = Path('../..')
MODELS_DIR = ROOT / 'models'
CHECKPOINT_PATH = MODELS_DIR / 'ppo_pacman'
CHECKPOINT_ZIP = CHECKPOINT_PATH.with_suffix('.zip')

N_STACK = 4
N_ENVS = 4
N_STEPS = 512
BATCH_SIZE = 512
TRAIN_STEPS = 50_000
MAX_STEPS = 16000

def make_env(seed: int, include_completion_plane: bool, include_frightened_plane: bool):
    def _factory():
        env = PacmanGridEnv(
            seed=seed,
            max_steps=MAX_STEPS,
            human_fair=True,
            include_completion_plane=include_completion_plane,
            include_frightened_plane=include_frightened_plane,
            near_miss_penalty=-12.0,
            late_endgame_fail_penalty=-4.0,
        )
        env = wrap_with_action_masker(env)
        return Monitor(env)
    return _factory

def build_vec_env(n_envs: int, include_completion_plane: bool, include_frightened_plane: bool):
    env_fns = [
        make_env(i, include_completion_plane, include_frightened_plane)
        for i in range(n_envs)
    ]
    venv = DummyVecEnv(env_fns)
    venv = VecMonitor(venv)
    return VecFrameStack(venv, n_stack=N_STACK, channels_order='first')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
print('checkpoint exists:', CHECKPOINT_ZIP.exists())

device: cuda
checkpoint exists: True


In [2]:
# 2) Integracja z MLOps (MLflow)
MLFLOW_DB = (ROOT / 'mlruns' / 'mlflow.db').resolve()
tracking_uri = f'sqlite:///{MLFLOW_DB}'
mlflow.set_tracking_uri(tracking_uri)

print('MLflow tracking URI:', tracking_uri)
print('MLflow DB exists:', MLFLOW_DB.exists())

# Szybki test polaczenia z MLOps
with MLflowLogger(experiment_name='rl_training', run_name='pipeline_bootstrap') as logger:
    logger.log_params({'source_notebook': '04_pipeline_integration', 'stage': 'mlops_bootstrap'})
    logger.log_metric('pipeline_bootstrap_ok', 1.0)

print('MLOps bootstrap run zapisany.')

MLflow tracking URI: sqlite:////home/mwrona/PAC-MAN-AI/mlruns/mlflow.db
MLflow DB exists: True
MLOps bootstrap run zapisany.


In [ ]:
# 3) RL zintegrowany z MLOps od startu
include_completion_plane = True
include_frightened_plane = True
if CHECKPOINT_ZIP.exists():
    include_completion_plane, include_frightened_plane = include_flags_from_checkpoint(str(CHECKPOINT_ZIP), N_STACK)

print('include_completion_plane:', include_completion_plane)
print('include_frightened_plane:', include_frightened_plane)

train_env = build_vec_env(
    n_envs=N_ENVS,
    include_completion_plane=include_completion_plane,
    include_frightened_plane=include_frightened_plane,
)

create_kwargs = dict(
    learning_rate=1e-4,
    n_steps=N_STEPS,
    batch_size=BATCH_SIZE,
    ent_coef=0.02,
    device=device,
    tensorboard_log=str(ROOT / 'logs' / 'tensorboard'),
)

if CHECKPOINT_ZIP.exists():
    model = load_trainable_model(
        str(CHECKPOINT_PATH),
        train_env,
        use_maskable=True,
        device=device,
        create_kwargs=create_kwargs,
    )
    print('Loaded checkpoint:', CHECKPOINT_ZIP)
else:
    model = create_maskable_ppo(train_env, **create_kwargs)
    print('Created fresh MaskablePPO model')

with MLflowLogger(experiment_name='rl_training', run_name='pipeline_rl_mlops') as logger:
    logger.log_params({
        'source_notebook': '04_pipeline_integration',
        'algorithm': 'MaskablePPO',
        'n_envs': N_ENVS,
        'n_stack': N_STACK,
        'train_steps': TRAIN_STEPS,
        'max_steps': MAX_STEPS,
        'include_completion_plane': include_completion_plane,
        'include_frightened_plane': include_frightened_plane,
        'near_miss_penalty': -12.0,
        'late_endgame_fail_penalty': -4.0,
    })

    model.learn(total_timesteps=TRAIN_STEPS, progress_bar=False)

    # szybka ewaluacja po treningu
    eval_env = build_vec_env(
        n_envs=1,
        include_completion_plane=include_completion_plane,
        include_frightened_plane=include_frightened_plane,
    )
    ep_returns = []
    clears = 0
    n_eval_episodes = 5

    for _ in range(n_eval_episodes):
        obs = eval_env.reset()
        done = False
        ep_return = 0.0
        while not done:
            action, _ = predict_action(model, obs, eval_env, deterministic=True)
            obs, rewards, dones, infos = eval_env.step(action)
            ep_return += float(rewards[0])
            done = bool(dones[0])
            if done:
                info = infos[0]
                ep_returns.append(ep_return)
                clears += int(info.get('level_clears', 0) > 0)

    eval_env.close()

    mean_return = float(np.mean(ep_returns)) if ep_returns else 0.0
    clear_rate = clears / max(1, n_eval_episodes)

    logger.log_metrics({
        'post_train_eval_mean_return': mean_return,
        'post_train_eval_clear_rate': clear_rate,
        'total_timesteps': float(model.num_timesteps),
    })

    model.save(str(CHECKPOINT_PATH))

print('Training done. Saved checkpoint:', CHECKPOINT_ZIP)
print('Post-train eval mean return:', round(mean_return, 3))
print('Post-train eval clear rate:', round(clear_rate, 3))

In [ ]:
# 4) Statystyki - podglad runow i metryk z MLOps
client = mlflow.tracking.MlflowClient(tracking_uri=tracking_uri)
experiment = client.get_experiment_by_name('rl_training')

if experiment is None:
    print("Brak eksperymentu 'rl_training'.")
else:
    runs = client.search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=['attributes.start_time DESC'],
        max_results=20,
    )

    rows = []
    for r in runs:
        rows.append({
            'run_name': r.data.tags.get('mlflow.runName', r.info.run_id[:8]),
            'status': r.info.status,
            'start_time': pd.to_datetime(r.info.start_time, unit='ms'),
            'total_timesteps': r.data.metrics.get('total_timesteps', np.nan),
            'eval_mean_return': r.data.metrics.get('eval_mean_return', np.nan),
            'eval_clear_rate': r.data.metrics.get('eval_level_clear_rate', np.nan),
            'post_eval_return': r.data.metrics.get('post_train_eval_mean_return', np.nan),
            'post_eval_clear_rate': r.data.metrics.get('post_train_eval_clear_rate', np.nan),
        })

    df = pd.DataFrame(rows)
    display(df.head(10))

    if not df.empty:
        plot_df = df.dropna(subset=['total_timesteps', 'eval_mean_return']).copy()
        if not plot_df.empty:
            plot_df = plot_df.sort_values('total_timesteps')
            ax = plot_df.plot(
                x='total_timesteps',
                y='eval_mean_return',
                marker='o',
                figsize=(8, 4),
                title='MLflow: eval_mean_return vs total_timesteps',
                grid=True,
            )
            ax.set_xlabel('total_timesteps')
            ax.set_ylabel('eval_mean_return')
